In [1]:
!pip install paho-mqtt

In [2]:
import paho.mqtt.client as mqtt
import time

# === HiveMQ Public Broker ===
BROKER = "broker.hivemq.com"
PORT = 1883
DATA_TOPIC = "microbit/data"
CMD_TOPIC = "microbit/commands"

In [3]:
#Fake process
def process_with_llm(data: str) -> str:
    print(f"LLM received: {data}")
    # Replace with your real LLM call later
    if "temp" in data.lower() and int(data.split(":")[-1]) > 30:
        return "FAN_ON"
    elif "light" in data.lower() and int(data.split(":")[-1]) < 50:
        return "LED_ON"
    else:
        return "LED_OFF"

In [4]:
def on_connect(client, userdata, flags, rc, properties=None):
    print(f"✅ Connected to HiveMQ with result code {rc}")
    client.subscribe(DATA_TOPIC)
    print(f"Subscribed to {DATA_TOPIC}")

def on_message(client, userdata, msg):
    data = msg.payload.decode().strip()
    print(f"📥 Received from micro:bit: {data}")
    command = process_with_llm(data)
    print(f"📤 Sending command: {command}")
    client.publish(CMD_TOPIC, command)

In [5]:
client = mqtt.Client(
    mqtt.CallbackAPIVersion.VERSION2,
    client_id="colab_llm"
)

client.on_connect = on_connect
client.on_message = on_message

# Optional: better reconnection
client.reconnect_delay_set(min_delay=1, max_delay=10)

try:
    client.connect(BROKER, PORT, 60)
    client.loop_start()
    print("🚀 Colab MQTT client started with HiveMQ")
except Exception as e:
    print(f"❌ Connection failed: {e}")

🚀 Colab MQTT client started with HiveMQ


In [6]:
# H1 - Activation Logic with MQTT
import ipywidgets as w
from IPython.display import display
import time
import paho.mqtt.client as mqtt   # already installed

# ====================== MQTT CONFIG ======================
BROKER = "broker.hivemq.com"
PORT = 1883
CMD_TOPIC = "microbit/commands"     # ← same as your existing setup

# ====================== FEAR LOGIC ======================
def send_fear_command():
    """Send FEAR trigger to the local machine via MQTT"""
    try:
        client.publish(CMD_TOPIC, "FEAR", qos=1)
        print("📤 FEAR command published to local machine via MQTT")
    except Exception as e:
        print(f"❌ Failed to send FEAR command: {e}")

def on_fear(prob, threshold=0.7, cd_ms=1500):
    """
    Translates model probability into a hardware trigger.
    Sends command over MQTT instead of calling local function.
    """
    if not hasattr(on_fear, 't0'):
        on_fear.t0 = 0

    now = time.time() * 1000

    if prob >= threshold and (now - on_fear.t0) > cd_ms:
        send_fear_command()
        on_fear.t0 = now
        print(f'FEAR Command Sent! (Model State: {prob:.2f})')

# ====================== UI ======================
thr = w.FloatSlider(value=0.7, min=0, max=1, step=0.01, description='FEAR thr')
b = w.Button(description='Test FEAR', button_style='danger')

def _manual_trigger_click(_):
    send_fear_command()
    print("Manual FEAR Trigger Sent via UI (MQTT)")

b.on_click(_manual_trigger_click)

# Display the dashboard
display(w.VBox([thr, b]))

import time
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    client.loop_stop()

✅ Connected to HiveMQ with result code Success
Subscribed to microbit/data
📥 Received from micro:bit: temp:35
LLM received: temp:35
📤 Sending command: FAN_ON
📥 Received from micro:bit: light:40
LLM received: light:40
📤 Sending command: LED_ON
📥 Received from micro:bit: temp:25
LLM received: temp:25
📤 Sending command: LED_OFF
📥 Received from micro:bit: button:pressed
LLM received: button:pressed
📤 Sending command: LED_OFF
📥 Received from micro:bit: temp:42
LLM received: temp:42
📤 Sending command: FAN_ON
📤 FEAR command published to local machine via MQTT
Manual FEAR Trigger Sent via UI (MQTT)
📤 FEAR command published to local machine via MQTT
Manual FEAR Trigger Sent via UI (MQTT)
